# GPU pipeline: QLoRA training, LLM evaluation and serving benchmark

Runs every GPU step of the repo on one NVIDIA GPU and packs the results into `results.zip`.

**Kaggle:** Settings → Accelerator → *GPU T4 x2* (one is used), Settings → Internet → *On* (needs a phone-verified account).
**RunPod / Lambda / other:** open Jupyter on a GPU pod with CUDA 12 and Python 3.11+.

Run the cells in order. Each step writes JSON under `results/` and a log under `logs/`. If a step fails, the later
steps that don't depend on it can still run; send back `results.zip` either way, because the logs say what failed.
Expected time on a T4: about 4 to 6 hours in total, most of it training. On an L4 or A10, about half that.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
import os
REPO = "https://github.com/juhidixit/vehicle-complaints-lora-vllm"
if not os.path.exists("vehicle-complaints-lora-vllm"):
    !git clone {REPO}
%cd vehicle-complaints-lora-vllm
!mkdir -p logs
!git log --oneline -1

In [ ]:
# vLLM pins its own torch build; this replaces the preinstalled one (takes a few minutes).
!pip install -q -e ".[gpu,dev]" 2>&1 | tail -5
!python -c "import torch, vllm; print(torch.__version__, torch.cuda.get_device_name(0), vllm.__version__)"

**Price used for cost per 1,000 classifications.** Enter what this GPU costs per hour (what you pay on RunPod,
or the on-demand list price of the same GPU if you're on free Kaggle) and where the number comes from.

In [ ]:
GPU_NAME = "NVIDIA T4 (Kaggle)"      # edit
GPU_USD_PER_HOUR = 0.35               # edit: price per hour you are using as the reference
GPU_PRICE_SOURCE = "edit me: e.g. RunPod T4 community price, checked 2026-11-20"
CPU_USD_PER_HOUR = None               # optional: price of a 4 vCPU VM, for the TF-IDF point on the cost chart
CPU_PRICE_SOURCE = None

import yaml
p = yaml.safe_load(open("bench/prices.yaml"))
p["gpu"].update(name=GPU_NAME, usd_per_hour=GPU_USD_PER_HOUR, source=GPU_PRICE_SOURCE)
p["cpu"].update(usd_per_hour=CPU_USD_PER_HOUR, source=CPU_PRICE_SOURCE)
yaml.safe_dump(p, open("bench/prices.yaml", "w"), sort_keys=False)
print(open("bench/prices.yaml").read())

## 1. Data: download today's NHTSA file and rebuild the committed splits

In [ ]:
!make data-from-ids 2>&1 | tee logs/data.log | tail -20

## 2. Prompting baselines (zero-shot and 5-shot, base model)

In [ ]:
!make llm-baselines 2>&1 | tee logs/llm_baselines.log | grep -E 'macro-F1|Error|error' | tail -20

## 3. QLoRA fine-tuning
One seed by default. If time allows, run `SEEDS="0 1 2"` instead (about 3x the time).

In [ ]:
SEEDS = "0"
!make train SEEDS="{SEEDS}" 2>&1 | tee logs/train.log | tail -30

## 4. Merge the adapter and score the fine-tuned model

In [ ]:
!make eval SEEDS="{SEEDS}" 2>&1 | tee logs/eval.log | grep -E 'macro-F1|Error|error' | tail -20

## 5. 4-bit (GPTQ W4A16) quantization and its accuracy

In [ ]:
!make quantize SEEDS="{SEEDS}" 2>&1 | tee logs/quantize.log | tail -20

## 6. Serving benchmark: HF vs vLLM offline, then vLLM server load tests

In [ ]:
!make bench SEEDS="{SEEDS}" 2>&1 | tee logs/bench.log | tail -40

## 7. Tables, charts and the results archive

In [ ]:
!make report 2>&1 | tail -3
import os
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("..")
!zip -qr {out_dir}/results.zip results logs README.md bench/prices.yaml
print("Download:", os.path.join(out_dir, "results.zip"))

## 8. Optional: publish the adapter on Hugging Face
Add a **write** token as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets), or set it in the environment on other hosts.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("No Kaggle secret:", e)
if os.environ.get("HF_TOKEN"):
    !python train/push_adapter.py --adapter outputs/adapters/seed0 --repo juhidixit/qwen2.5-1.5b-nhtsa-complaints-lora